# build User Data

In [ ]:
import json
# 建構使用者
# 使用者內容包括
class User:
    def __init__(self, username, password, email):
        self.username = username
        self.password = password
        self.email = email
# 輸入：num_users：數量, startNumber：從幾號開始取名，取名的方式
def create_users_json(num_users,startNumber):
    users = []
    startNumber-=1
    for i in range(1, num_users+1):
        user = User(str(i+startNumber), str(i+startNumber), f"redacted-user@example.com")
        users.append(user.__dict__)
    return json.dumps(users)

In [ ]:
# 建構 3 個使用者的 JSON 格式變數
json_var = create_users_json(10,1)
print(json_var)

# insert_many data to mongodb

In [ ]:
import pymongo

def insert_data_to_mongodb(data, uri='localhost:27017', database='sample_users', collection='accounts'):
    client = pymongo.MongoClient(uri)
    db = client[database]
    col = db[collection]
    try:
        col.insert_many(data)
        print("Data insertion successful!")
        return True
    except Exception as e:
        print(f"Data insertion failed: {e}")
        return False
    finally:
        client.close()

In [ ]:
# 因為 create_users_json() 返回的是以下 JSON 字串
data_str = create_users_json(1000,1)
# 將 JSON 字串解析為 Python 對象
data = json.loads(data_str)
# insert_many to mongodb
insert_data_to_mongodb(data)

# mongodb opration

In [ ]:
import pymongo

class MongoDB:
    def __init__(self, uri='localhost:27017', database='sample_users', collection='accounts'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.client = pymongo.MongoClient(self.uri)
        self.db = self.client[self.database]
        self.col = self.db[self.collection]

    def insert_data(self, data):
        try:
            result = self.col.insert_many(data)
            print("Data insertion successful!")
            return result.inserted_ids
        except Exception as e:
            print(f"Data insertion failed: {e}")
            return None

    def find_data(self, query={}, projection=None, limit=0):
        cursor = self.col.find(query, projection=projection, limit=limit)
#         return cursor
        return [doc for doc in cursor]
    def close(self):
        self.client.close()
    def get_max_EventID0308(self):
        query = {"EventID0308": {"$exists": True}}
        projection = {"EventID0308": 1, "_id": 0}
        cursor = self.col.find(query, projection).sort("EventID0308", pymongo.DESCENDING).limit(1)
        result = list(cursor)
        if result:
            max_event_id = result[0]["EventID0308"]
            return max_event_id
        else:
            print("No EventID0308 found in collection")
            return Non


In [ ]:
    def take_idAndEventID0308FromOrinArticle():
        mongo = MongoDB('localhost:27017', 'sample_event_DB', collection='sample_event_Coll_copy')
        data = mongo.find_data({},{'_id':1,'EventID0308':1},3)
        # max_EventID0308=mongo.get_max_EventID0308()
        mongo.close()

In [ ]:
mongo = MongoDB()
users_id = mongo.find_data({},{'_id':1})
print(users_id)

In [ ]:
mongo = MongoDB('localhost:27017', 'sample_event_DB', collection='sample_event_Coll_copy')
data = mongo.find_data({'EventID0308':1},{'_id':1,'EventID0308':1})
# max_EventID0308=mongo.get_max_EventID0308()
mongo.close()
print(data)

# build the User Browsing History

In [ ]:
import random
import pymongo

class JSONGenerator_UserBrowsingHistory:
    def __init__(self, uri='localhost:27017', database='sample_event_DB', collection='sample_event_Coll_copy'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.client = pymongo.MongoClient(self.uri)
        self.db = self.client[self.database]
        self.col = self.db[self.collection]
    
    def generate_json(self):
        data = {}
        # get random document from collection
        cursor = self.col.find({}, {'_id': 1, 'EventID0308': 1})
        doc = random.choice([d for d in cursor])
        data['Event_id'] = str(doc['_id'])
        data['EventID'] = str(doc['EventID0308'])
        # generate random view time
        data['view_time'] = '2022-04-01T12:00:00.000Z'
        # generate random browsing duration
        data['browsing_duration'] = random.randint(0, 30)
        # generate random topic
        data['topic'] = random.randint(0, 3)
        return data
    
    def find_data(self, query={}, projection=None, limit=0):
        cursor = self.col.find(query, projection=projection, limit=limit)
        return [doc for doc in cursor]
    
    def close(self):
        self.client.close()

In [ ]:
# importing ObjectId from bson library
from bson.objectid import ObjectId
json_generator = JSONGenerator_UserBrowsingHistory()
json_data = json_generator.generate_json()
print(f'make a json_data: {json_data}\n')
print(f'json_data type is: {type(json_data)}\n')
print(f'take the Event_id: {json_data["Event_id"]}\n')
print(f'Event_id type is: {type(json_data["Event_id"])}\n')
print(f'take the EventID: {json_data["EventID"]}\n')
print(f'EventID type is: {type(json_data["EventID"])}\n')
query = {"EventID0308": int(json_data["EventID"])} # must be int() when you find with "EventID" !!!
result = json_generator.find_data(query,{'_id':1,'EventID0308':1})
print('result with EventID0308: \n',result)
query = {"_id":  ObjectId(json_data["Event_id"])}
result = json_generator.find_data(query,{'_id':1,'EventID0308':1})  # must be ObjectId() when you find with "EventID" !!!
print('result with _id: \n',result)
print(f'ObjectId type is: {type(ObjectId(json_data["Event_id"]))}')# It is <class 'bson.objectid.ObjectId'> !!! special format
query = {"_id":  ObjectId(json_data["Event_id"])}
result = json_generator.find_data(query) 
print('all result: \n',result)